# Notebook 12 — Site-level paired bootstrap (robustness analysis)

**Purpose (supervisor decision 4A).** Estimate the uncertainty of the reported test scores that comes from the finite test set, without retraining any model.

**Method.** For each monitoring network, the test monitoring sites are resampled with replacement (B = 2,000 replicates; independent random streams per network, spawned from seed 42). All sequences of a drawn site are kept together, because sequences from the same site are not independent. In every replicate, all models and the persistence baseline are evaluated on the **same** resampled sequences (paired bootstrap), so differences between models can be assessed directly. Present-class Macro-F1 is computed with the class list **fixed** to the classes present in the full test set of that network.

**Inputs to attach (Add Input):**
1. The **original** Notebook 02 output (June version, `tensors/<client>_stage1_sequences.npz`) — the reference for sample count, labels, sites and dates.
2. The output of the **copy** of Notebook 02 that exported `persistence_export/persistence_<client>_test_preds.npz` (verified against the original tensors when it was created).
3. The **original** outputs of Notebooks 03 (GRU), 04 (LSTM), 05 (Transformer) and 08 (classical ML) — the versions that produced the values reported in the manuscript.

Section 1 stops the notebook if the persistence file differs from the original tensors in sample count, labels, site order or target-date order. Section 2 stops it if any prediction file is misaligned or does not reproduce the reported value. Deep-learning prediction files contain no site identifiers, so their alignment is checked through the label sequence and the reproduced scores.

In [29]:
# 0 — Setup and file index (normal files and files inside ZIP archives)
import os, io, zipfile, shutil, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", None)

ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
B, SEED, K = 2000, 42, 5
OUT = Path("/kaggle/working/bootstrap_outputs")
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)

normal_index, zip_index = {}, {}
for root in [Path("/kaggle/input")]:
    for dirpath, _, files in os.walk(root):
        for fn in files:
            p = Path(dirpath) / fn
            normal_index.setdefault(fn, []).append(p)
            if p.suffix.lower() == ".zip":
                try:
                    with zipfile.ZipFile(p) as z:
                        for name in z.namelist():
                            if not name.endswith("/"):
                                zip_index.setdefault(Path(name).name, []).append((p, name))
                except Exception:
                    pass

def _open(fn):
    hits = [("file", p, None) for p in normal_index.get(fn, [])] + [("zip", zp, m) for zp, m in zip_index.get(fn, [])]
    if not hits:
        raise FileNotFoundError(fn)
    if len(hits) > 1:
        print(f"  note: {fn} found {len(hits)} times; using {hits[0][1]}")
    typ, p, member = hits[0]
    if typ == "file":
        return open(p, "rb").read()
    with zipfile.ZipFile(p) as z:
        return z.read(member)

load_npz = lambda fn: np.load(io.BytesIO(_open(fn)), allow_pickle=False)
load_csv = lambda fn: pd.read_csv(io.BytesIO(_open(fn)))
print("Indexed files:", len(normal_index), "| files inside ZIPs:", len(zip_index))

Indexed files: 362 | files inside ZIPs: 0


In [30]:
# 1 — Reference test data from the ORIGINAL Notebook 02 tensors, and the verified persistence predictions
def all_copies(fn):
    hits = [("file", p, None) for p in normal_index.get(fn, [])] + [("zip", zp, m) for zp, m in zip_index.get(fn, [])]
    out = []
    for typ, p, member in hits:
        raw = open(p, "rb").read() if typ == "file" else zipfile.ZipFile(p).read(member)
        out.append((str(p) + ("" if member is None else "!" + member), np.load(io.BytesIO(raw), allow_pickle=False)))
    return out

month = lambda a: pd.to_datetime(pd.Series(np.asarray(a).astype(str))).dt.to_period("M").astype(str).to_numpy()

ref = {}
for c in ACTIVE_CLIENTS:
    copies = all_copies(f"{c.lower()}_stage1_sequences.npz")
    assert copies, f"Original Notebook 02 tensors for {c} not found"
    t0 = copies[0][1]
    # If several copies are attached (original + regenerated), they must be identical; otherwise STOP.
    for path, t in copies[1:]:
        for k in ["y_test", "site_ids_test", "target_dates_test"]:
            assert np.array_equal(t[k].astype(str), t0[k].astype(str)), f"{c}: {k} differs between attached tensor copies ({path})"
        assert np.allclose(t["X_test"], t0["X_test"], atol=1e-5), f"{c}: X_test differs between attached tensor copies ({path})"
    ref[c] = {"y": t0["y_test"].astype(int), "site": t0["site_ids_test"].astype(str), "month": month(t0["target_dates_test"])}

    p = load_npz(f"persistence_{c.lower()}_test_preds.npz")
    assert len(p["y_pred"]) == len(ref[c]["y"]), f"{c}: persistence file has a different number of sequences"
    assert np.array_equal(p["y_true"].astype(int), ref[c]["y"]), f"{c}: persistence labels differ from the original tensors"
    assert np.array_equal(p["site_ids"].astype(str), ref[c]["site"]), f"{c}: persistence site order differs from the original tensors"
    assert np.array_equal(month(p["target_dates"]), ref[c]["month"]), f"{c}: persistence target dates differ from the original tensors"
    ref[c]["preds"] = {"Persistence": p["y_pred"].astype(int)}
    print(f"{c:8s} test sequences = {len(ref[c]['y']):,} | sites = {len(np.unique(ref[c]['site'])):,} | "
          f"classes = {sorted(np.unique(ref[c]['y']).tolist())} | tensor copies found = {len(copies)}")

England  test sequences = 11,942 | sites = 1,760 | classes = [0, 1, 2, 3, 4] | tensor copies found = 1
Ireland  test sequences = 5,887 | sites = 255 | classes = [0, 1, 2] | tensor copies found = 1
USA      test sequences = 1,979 | sites = 77 | classes = [0, 1] | tensor copies found = 1


In [31]:
# 2 — Load the saved test predictions of every reported model and verify them
CLASSICAL = {"Logistic Regression": "logistic_regression", "Random Forest": "random_forest", "XGBoost": "xgboost", "CatBoost": "catboost"}
DEEP_LOCAL   = {"GRU": "local_{c}", "LSTM": "local_lstm_{c}", "Transformer": "local_transformer_{c}"}
DEEP_CENTRAL = {"GRU": "centralized_{c}", "LSTM": "centralized_lstm_{c}", "Transformer": "centralized_transformer_{c}"}
FEDERATED = {"GRU FedAvg v3": "fedavg_v3_{c}", "GRU FedAvg v2": "fedavg_v2_{c}", "LSTM FedProx v1": "lstm_fedprox_v1_{c}",
             "LSTM FedAvg v3": "lstm_fedavg_v3_{c}", "Transformer FedProx v1": "transformer_fedprox_v1_{c}"}

# Present-class Macro-F1 reported in the manuscript (England, Ireland, USA); None = not reported for that network
EXPECTED = {
 "Local Logistic Regression": (0.6567, 0.4118, 0.7421), "Local Random Forest": (0.6832, 0.5030, 0.7792),
 "Local XGBoost": (0.7145, 0.4995, 0.7889), "Local CatBoost": (0.6973, 0.4933, 0.7848),
 "Local GRU": (0.6941, 0.4864, 0.7909), "Local LSTM": (0.6735, 0.4907, 0.7739), "Local Transformer": (0.7091, 0.4955, 0.7896),
 "Centralized Logistic Regression": (0.5212, 0.2626, 0.4596), "Centralized Random Forest": (0.6636, 0.4988, 0.7727),
 "Centralized XGBoost": (0.6699, 0.5381, 0.7735), "Centralized CatBoost": (0.6561, 0.5006, 0.7433),
 "Centralized GRU": (0.6910, 0.5130, 0.7718), "Centralized LSTM": (0.6664, 0.5142, 0.7559), "Centralized Transformer": (0.6719, 0.5206, 0.7810),
 "Federated GRU FedAvg v3": (0.2546, 0.4098, 0.5906), "Federated LSTM FedProx v1": (0.2427, 0.4294, 0.6258),
 "Federated Transformer FedProx v1": (0.1845, 0.3936, 0.4747), "Federated GRU FedAvg v2": (0.4915, None, None),
 "Federated LSTM FedAvg v3": (None, None, 0.6785), "Persistence": (0.6970, 0.5696, 0.8391),
}

def macro_f1_fixed(y_true, y_pred, classes):
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm)[classes].astype(float)
    fp = cm[:, classes].sum(0) - tp
    fn = cm[classes, :].sum(1) - tp
    den = 2 * tp + fp + fn
    return np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0).mean()

def add(c, name, y_true, y_pred, sites=None, dates=None):
    assert len(y_pred) == len(ref[c]["y"]), f"{name} / {c}: number of predictions differs"
    assert np.array_equal(np.asarray(y_true).astype(int), ref[c]["y"]), f"{name} / {c}: y_true not aligned with the original tensors"
    if sites is not None:
        assert np.array_equal(np.asarray(sites).astype(str), ref[c]["site"]), f"{name} / {c}: site order differs"
    if dates is not None:
        assert np.array_equal(month(dates), ref[c]["month"]), f"{name} / {c}: target-date order differs"
    ref[c]["preds"][name] = np.asarray(y_pred).astype(int)

for c in ACTIVE_CLIENTS:
    cl = c.lower()
    for nice, key in CLASSICAL.items():
        d = load_csv(f"local_{key}_{cl}_to_{cl}_test_predictions.csv"); add(c, f"Local {nice}", d.y_true, d.y_pred, d.site_id, d.target_date)
        d = load_csv(f"centralized_{key}_pooled_to_{cl}_test_predictions.csv"); add(c, f"Centralized {nice}", d.y_true, d.y_pred, d.site_id, d.target_date)
    for nice, pat in DEEP_LOCAL.items():
        d = load_npz(pat.format(c=cl) + "_test_preds.npz"); add(c, f"Local {nice}", d["y_true"], d["y_pred"])
    for nice, pat in DEEP_CENTRAL.items():
        d = load_npz(pat.format(c=cl) + "_test_preds.npz"); add(c, f"Centralized {nice}", d["y_true"], d["y_pred"])
    for nice, pat in FEDERATED.items():
        d = load_npz(pat.format(c=cl) + "_test_preds.npz"); add(c, f"Federated {nice}", d["y_true"], d["y_pred"])

rows, bad = [], []
for c_i, c in enumerate(ACTIVE_CLIENTS):
    classes = np.unique(ref[c]["y"])
    for name, yp in ref[c]["preds"].items():
        f1 = macro_f1_fixed(ref[c]["y"], yp, classes)
        exp = EXPECTED.get(name, (None,) * 3)[c_i]
        ok = None if exp is None else bool(abs(round(float(f1), 4) - exp) < 1e-4)
        rows.append({"Client": c, "Model": name, "Macro_F1_recomputed": round(f1, 4), "Reported": exp, "Match": ok})
        if ok is False: bad.append((c, name, round(f1, 4), exp))
check_df = pd.DataFrame(rows)
display(check_df)
check_df.to_csv(OUT / "00_reproduction_check.csv", index=False)
if bad:
    raise RuntimeError(f"{len(bad)} prediction file(s) do not reproduce the reported values (wrong notebook version attached?): {bad}")
print("All loaded predictions reproduce the reported values.")

,Client,Model,Macro_F1_recomputed,Reported,Match
0,England,Persistence,0.6970,0.6970,True
1,England,Local Logistic Regression,0.6567,0.6567,True
2,England,Centralized Logistic Regression,0.5212,0.5212,True
3,England,Local Random Forest,0.6832,0.6832,True
4,England,Centralized Random Forest,0.6636,0.6636,True
5,England,Local XGBoost,0.7145,0.7145,True
6,England,Centralized XGBoost,0.6699,0.6699,True
7,England,Local CatBoost,0.6973,0.6973,True
8,England,Centralized CatBoost,0.6561,0.6561,True
9,England,Local GRU,0.6941,0.6941,True


All loaded predictions reproduce the reported values.


In [32]:
# 3 — Paired site-level bootstrap
# Independent random streams per country (spawned from one seed), so that replicates can be
# combined across countries as independent draws
STREAMS = dict(zip(ACTIVE_CLIENTS, np.random.SeedSequence(SEED).spawn(len(ACTIVE_CLIENTS))))

def site_bootstrap(c):
    y, site, preds = ref[c]["y"], ref[c]["site"], ref[c]["preds"]
    classes = np.unique(y)
    codes, uniq = pd.factorize(pd.Series(site))
    order = np.argsort(codes, kind="stable")
    bounds = np.searchsorted(codes[order], np.arange(len(uniq) + 1))
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniq))]
    rng = np.random.default_rng(STREAMS[c])
    names = list(preds); res = np.empty((B, len(names))); missing = 0
    for b in range(B):
        idx = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        yt = y[idx]
        missing += int(any((yt == k).sum() == 0 for k in classes))
        for j, n in enumerate(names):
            res[b, j] = macro_f1_fixed(yt, preds[n][idx], classes)
    return pd.DataFrame(res, columns=names), missing, len(uniq)

boot = {}
for c in ACTIVE_CLIENTS:
    boot[c], miss, n_sites = site_bootstrap(c)
    print(f"{c:8s} B = {B} replicates over {n_sites:,} sites | replicates with a missing class: {miss}")

England  B = 2000 replicates over 1,760 sites | replicates with a missing class: 1
Ireland  B = 2000 replicates over 255 sites | replicates with a missing class: 0
USA      B = 2000 replicates over 77 sites | replicates with a missing class: 0


In [33]:
# 4 — 95% percentile intervals for every model
ci_rows = []
for c in ACTIVE_CLIENTS:
    classes = np.unique(ref[c]["y"])
    for n in boot[c].columns:
        lo, hi = boot[c][n].quantile([0.025, 0.975])
        ci_rows.append({"Client": c, "Model": n, "Macro_F1": round(macro_f1_fixed(ref[c]["y"], ref[c]["preds"][n], classes), 4),
                        "CI95_low": round(lo, 4), "CI95_high": round(hi, 4)})
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT / "01_macro_f1_confidence_intervals.csv", index=False)
display(ci_df.pivot(index="Model", columns="Client", values=["Macro_F1", "CI95_low", "CI95_high"]).round(4))

Macro_F1                 CI95_low                 CI95_high                
Client                            England Ireland     USA  England Ireland     USA   England Ireland     USA
Model                                                                                                       
Centralized CatBoost               0.6561  0.5006  0.7433   0.5899  0.4696  0.6125    0.7057  0.5234  0.8175
Centralized GRU                    0.6910  0.5130  0.7718   0.5925  0.4850  0.6664    0.7461  0.5365  0.8366
Centralized LSTM                   0.6664  0.5142  0.7559   0.5799  0.4863  0.6447    0.6975  0.5360  0.8180
Centralized Logistic Regression    0.5212  0.2626  0.4596   0.4310  0.2433  0.4463    0.5878  0.2802  0.4734
Centralized Random Forest          0.6636  0.4988  0.7727   0.5937  0.4634  0.6150    0.7256  0.5247  0.8483
Centralized Transformer            0.6719  0.5206  0.7810   0.5853  0.4916  0.6547    0.7400  0.5451  0.8497
Centralized XGBoost                0.6699  0.5381  0.7735   0.5882  0.5057  0.6300    0.7351  0.5879  0.8473
Federated GRU FedAvg v2            0.4915  0.0798  0.1105   0.4254  0.0634  0.0825    0.5513  0.0978  0.1405
Federated GRU FedAvg v3            0.2546  0.4098  0.5906   0.2300  0.3431  0.5468    0.2744  0.4477  0.6300
Federated LSTM FedAvg v3           0.1842  0.4092  0.6785   0.1587  0.3748  0.5863    0.2072  0.4406  0.7452
Federated LSTM FedProx v1          0.2427  0.4294  0.6258   0.2215  0.3555  0.5738    0.2610  0.4826  0.6719
Federated Transformer FedProx v1   0.1845  0.3936  0.4747   0.1603  0.3341  0.4276    0.2076  0.4408  0.5200
Local CatBoost                     0.6973  0.4933  0.7848   0.5953  0.4709  0.6968    0.7538  0.5133  0.8528
Local GRU                          0.6941  0.4864  0.7909   0.5897  0.4642  0.6811    0.7404  0.5057  0.8566
Local LSTM                         0.6735  0.4907  0.7739   0.5929  0.4681  0.6627    0.7169  0.5101  0.8441
Local Logistic Regression          0.6567  0.4118  0.7421   0.5594  0.3752  0.6471    0.7117  0.4503  0.8002
Local Random Forest                0.6832  0.5030  0.7792   0.5992  0.4704  0.6091    0.7496  0.5278  0.8591
Local Transformer                  0.7091  0.4955  0.7896   0.6002  0.4723  0.6763    0.7519  0.5160  0.8582
Local XGBoost                      0.7145  0.4995  0.7889   0.6052  0.4774  0.6993    0.7573  0.5188  0.8530
Persistence                        0.6970  0.5696  0.8391   0.5735  0.4717  0.7646    0.7394  0.6555  0.8837

In [34]:
# 5 — Paired differences: every model minus persistence
diff_rows = []
for c in ACTIVE_CLIENTS:
    point = ci_df[ci_df.Client == c].set_index("Model")["Macro_F1"]
    for n in boot[c].columns:
        if n == "Persistence": continue
        d = boot[c][n] - boot[c]["Persistence"]
        lo, hi = d.quantile([0.025, 0.975])
        diff_rows.append({"Client": c, "Model": n, "Diff_vs_persistence": round(point[n] - point["Persistence"], 4),
                          "CI95_low": round(lo, 4), "CI95_high": round(hi, 4),
                          "Share_replicates_model_better": round((d > 0).mean(), 3),
                          "CI_excludes_0": bool(lo > 0 or hi < 0)})
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(OUT / "02_differences_vs_persistence.csv", index=False)
display(diff_df.sort_values(["Client", "Diff_vs_persistence"], ascending=[True, False]))

,Client,Model,Diff_vs_persistence,CI95_low,CI95_high,Share_replicates_model_better,CI_excludes_0
4,England,Local XGBoost,0.0175,-0.0257,0.0528,0.698,False
10,England,Local Transformer,0.0121,-0.0227,0.0406,0.753,False
6,England,Local CatBoost,0.0003,-0.0753,0.0549,0.455,False
8,England,Local GRU,-0.0029,-0.0554,0.0368,0.427,False
11,England,Centralized GRU,-0.0060,-0.0796,0.0454,0.405,False
2,England,Local Random Forest,-0.0138,-0.1152,0.0527,0.382,False
9,England,Local LSTM,-0.0235,-0.0799,0.0279,0.240,False
13,England,Centralized Transformer,-0.0251,-0.1282,0.0429,0.309,False
5,England,Centralized XGBoost,-0.0271,-0.1260,0.0380,0.304,False
12,England,Centralized LSTM,-0.0306,-0.0536,0.0134,0.060,False


In [35]:
# 6 — Paired differences used in the manuscript text
PAIRS = {  # (label, model A, model B) per client: A minus B
 "England": [("Local XGBoost - Local Transformer", "Local XGBoost", "Local Transformer"),
             ("Best local - best centralized (XGBoost - GRU)", "Local XGBoost", "Centralized GRU"),
             ("Best centralized - best federated (GRU - GRU FedAvg v2)", "Centralized GRU", "Federated GRU FedAvg v2")],
 "Ireland": [("Local XGBoost - Local Transformer", "Local XGBoost", "Local Transformer"),
             ("Best centralized - best local (XGBoost - Random Forest)", "Centralized XGBoost", "Local Random Forest"),
             ("Best local - best federated (Random Forest - LSTM FedProx v1)", "Local Random Forest", "Federated LSTM FedProx v1")],
 "USA":     [("Local XGBoost - Local Transformer", "Local XGBoost", "Local Transformer"),
             ("Best local - best centralized (GRU - Transformer)", "Local GRU", "Centralized Transformer"),
             ("Best centralized - best federated (Transformer - LSTM FedAvg v3)", "Centralized Transformer", "Federated LSTM FedAvg v3")],
}
pair_rows = []
for c, plist in PAIRS.items():
    point = ci_df[ci_df.Client == c].set_index("Model")["Macro_F1"]
    for label, a, b_ in plist:
        d = boot[c][a] - boot[c][b_]; lo, hi = d.quantile([0.025, 0.975])
        pair_rows.append({"Client": c, "Comparison": label, "Difference": round(point[a] - point[b_], 4),
                          "CI95_low": round(lo, 4), "CI95_high": round(hi, 4), "CI_excludes_0": bool(lo > 0 or hi < 0)})

# Mean across the three networks (replicates are independent between networks, so they are combined index-wise)
for a, b_ in [("Local XGBoost", "Local Transformer")]:
    da = sum(boot[c][a] for c in ACTIVE_CLIENTS) / 3; db = sum(boot[c][b_] for c in ACTIVE_CLIENTS) / 3
    d = da - db; lo, hi = d.quantile([0.025, 0.975])
    pa = ci_df[ci_df.Model == a].Macro_F1.mean(); pb = ci_df[ci_df.Model == b_].Macro_F1.mean()
    pair_rows.append({"Client": "Mean of 3", "Comparison": f"{a} - {b_}", "Difference": round(pa - pb, 4),
                      "CI95_low": round(lo, 4), "CI95_high": round(hi, 4), "CI_excludes_0": bool(lo > 0 or hi < 0)})
pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(OUT / "03_selected_paired_differences.csv", index=False)
display(pair_df)

,Client,Comparison,Difference,CI95_low,CI95_high,CI_excludes_0
0,England,Local XGBoost - Local Transformer,0.0054,-0.0487,0.0536,False
1,England,Best local - best centralized (XGBoost - GRU),0.0235,-0.0044,0.0637,False
2,England,Best centralized - best federated (GRU - GRU F...,0.1995,0.1444,0.2240,True
3,Ireland,Local XGBoost - Local Transformer,0.0040,-0.0053,0.0140,False
4,Ireland,Best centralized - best local (XGBoost - Rando...,0.0351,0.0107,0.0851,True
5,Ireland,Best local - best federated (Random Forest - L...,0.0736,0.0249,0.1430,True
6,USA,Local XGBoost - Local Transformer,-0.0007,-0.0415,0.0501,False
7,USA,Best local - best centralized (GRU - Transformer),0.0099,-0.0134,0.0466,False
8,USA,Best centralized - best federated (Transformer...,0.1025,0.0354,0.1510,True
9,Mean of 3,Local XGBoost - Local Transformer,0.0029,-0.0208,0.0280,False


In [36]:
# 7 — Save everything as one ZIP
shutil.make_archive("/kaggle/working/N12_bootstrap_outputs", "zip", OUT)
print("Saved:", sorted(p.name for p in OUT.iterdir()))
print("ZIP: /kaggle/working/N12_bootstrap_outputs.zip")

Saved: ['00_reproduction_check.csv', '01_macro_f1_confidence_intervals.csv', '02_differences_vs_persistence.csv', '03_selected_paired_differences.csv']
ZIP: /kaggle/working/N12_bootstrap_outputs.zip
